# PatchTST Baseline — Channel-Independent (all 6 datasets)

Trains **canonical PatchTST** (channel-independent: each channel = separate sample, shared weights),
d_model=512, e_layers=3, patch_len=16, stride 8, lookback 512, pred 48.

Channel-independent avoids the O(C²) attention blowup that OOMs on 321/862-channel data.
This matches the original PatchTST paper and is what `benchmark_standard.py` evaluates.

**Datasets**: ETTh1, ETTh2, ETTm1, exchange_rate, electricity, traffic (full standard protocol)
**Runtime**: ~10–30 min total on T4
**Resumable**: re-run all cells → picks up from last checkpoint (`*_resume.pt`)
**Drive**: checkpoints to `MyDrive/nanoforecast-baselines/patchtst/`


## Step 1 — GPU check + Drive mount

In [ ]:
import torch, sys, os, json, time, subprocess
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU.'
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/nanoforecast-baselines/patchtst'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


## Step 2 — Fetch benchmark code from GitHub (no upload needed)

The trainer + vendored PatchTST + harness were pushed to the repo (branch `v0.5`),
so we shallow-clone and import straight from the clone. No tarball to upload.

In [ ]:
# Shallow clone of the repo (benchmarks/ + benchmark_standard.py + nanoforecast/)
!git clone --depth 1 -b v0.5 https://github.com/eulogik/NanoForecast /content/NanoForecast

import os, sys
sys.path.insert(0, '/content/NanoForecast')
os.chdir('/content/NanoForecast')

need = ['benchmark_standard.py',
        'benchmarks/train_patchtst_ci.py',
        'benchmarks/tsl/PatchTST.py',
        'benchmarks/tsl/Embed.py']
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f'Missing files: {missing}'
print('Files OK:')
!ls benchmark_standard.py benchmarks/ benchmarks/tsl/


## Step 3 — Train all 6 datasets (channel-independent)

Run the cell below. Each dataset trains until early-stop (patience 3); checkpoints
and `*.json` metadata land in `MyDrive/nanoforecast-baselines/patchtst/` (via symlink).
Re-running resumes: datasets with a final `{ds}.pt` are skipped, partial runs resume.

In [ ]:
# Point the trainer's checkpoint dir at Drive
ckpt_dir = '/content/NanoForecast/benchmarks/checkpoints/patchtst'
os.makedirs(os.path.dirname(ckpt_dir), exist_ok=True)
if os.path.islink(ckpt_dir):
    os.unlink(ckpt_dir)
if not os.path.isdir(ckpt_dir):
    os.symlink(DRIVE_ROOT, ckpt_dir)
print('Checkpoints ->', DRIVE_ROOT)

from benchmarks.train_patchtst_ci import train_one

DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'exchange_rate', 'electricity', 'traffic']
for ds in DATASETS:
    print(f'\n=== Training {ds} ===')
    t0 = time.time()
    r = train_one(ds, 'cuda')
    if r:
        print(f'  {ds}: best_val_mse={r["best_val_mse"]:.6f} '
              f'epochs={r["epochs"]} ({time.time()-t0:.0f}s)')
    torch.cuda.empty_cache()

print('\n=== ALL DONE ===')
print('Checkpoints:', sorted(os.listdir(DRIVE_ROOT)))


## Step 4 — Verify + download checkpoints

After training, download all 6 pairs (`{ds}.pt`, `{ds}.json`) from
`MyDrive/nanoforecast-baselines/patchtst/` into your local
`~/Code/NanoForecast/benchmarks/checkpoints/patchtst/`, then run:

```
python3 benchmark_standard.py --models patchtst --datasets ETTh1,ETTh2,ETTm1,exchange_rate,electricity,traffic
```


In [ ]:
for f in sorted(os.listdir(DRIVE_ROOT)):
    sz = os.path.getsize(os.path.join(DRIVE_ROOT, f))
    print(f'  {f:30s} {sz/1024:8.1f} KB')

for ds in DATASETS:
    mp = os.path.join(DRIVE_ROOT, f'{ds}.json')
    if os.path.exists(mp):
        m = json.load(open(mp))
        print(f'{ds}: {m["n_vars"]} var(s), {m["epochs"]} epochs, '
              f'best_val_mse={m["best_val_mse"]:.6f}, {m["train_seconds"]}s, '
              f'CI={m.get("channel_independent")}')
    else:
        print(f'{ds}: NOT FOUND — train still running?')
